# Meeting Audio → Summary Pipeline

This notebook runs a local pipeline to:
1. Transcribe meeting audio using a local Whisper model (via `faster-whisper`)
2. Summarize the transcript using a local LLM (`qwen3.5:4b`) served by Ollama


## Prerequisites

- Python 3 in JupyterLab
- [Ollama](https://ollama.com/) installed and running locally
- Ability to run `ollama pull qwen3.5:4b`

The notebook will install missing Python packages and download model weights as needed.

In [ ]:
import importlib
import json
import subprocess
import sys
from pathlib import Path


def ensure_python_dependencies() -> None:
    """Install required Python packages if they are missing."""
    requirements = {
        "faster_whisper": "faster-whisper",
        "requests": "requests",
    }
    missing = [pkg_name for module_name, pkg_name in requirements.items() if importlib.util.find_spec(module_name) is None]

    if missing:
        print(f"Installing missing dependencies: {', '.join(missing)}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    else:
        print("Python dependencies already installed.")


ensure_python_dependencies()

In [ ]:
import requests
from faster_whisper import WhisperModel


def ensure_ollama_model(model_name: str = "qwen3.5:4b") -> None:
    """Pull the local LLM with Ollama if not present."""
    try:
        response = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            "Could not connect to Ollama at http://127.0.0.1:11434. Start Ollama and retry."
        ) from exc

    models = {item.get("name") for item in response.json().get("models", [])}
    if model_name in models:
        print(f"Ollama model '{model_name}' already available.")
        return

    print(f"Pulling Ollama model '{model_name}'...")
    subprocess.check_call(["ollama", "pull", model_name])


def load_whisper_model(model_size: str = "small", device: str = "cpu", compute_type: str = "int8") -> WhisperModel:
    """Load local Whisper model (downloads weights on first run)."""
    print(f"Loading Whisper model '{model_size}' on {device} ({compute_type})...")
    return WhisperModel(model_size, device=device, compute_type=compute_type)


def transcribe_audio(audio_path: str, stt_model: WhisperModel) -> str:
    """Transcribe an audio file into plain text."""
    audio_file = Path(audio_path).expanduser().resolve()
    if not audio_file.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_file}")

    segments, info = stt_model.transcribe(str(audio_file), beam_size=5, vad_filter=True)
    transcript = " ".join(segment.text.strip() for segment in segments if segment.text.strip())

    if not transcript:
        raise RuntimeError("No speech content could be transcribed from the audio file.")

    print(f"Transcription complete. Detected language: {info.language} (prob={info.language_probability:.2f})")
    return transcript


def summarize_transcript(
    transcript: str,
    llm_model: str = "qwen3.5:4b",
    request_timeout_seconds: int = 600,
) -> str:
    """Generate a meeting summary with action items using local Ollama model."""
    prompt = f"""You are an assistant that summarizes meeting transcripts.
Return:
1) A concise summary (5-10 bullet points)
2) Key decisions
3) Action items with owners if mentioned
4) Open questions

Transcript:
{transcript}
"""

    response = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={"model": llm_model, "prompt": prompt, "stream": False},
        timeout=request_timeout_seconds,
    )
    response.raise_for_status()

    data = response.json()
    summary = data.get("response", "").strip()
    if not summary:
        raise RuntimeError("LLM returned an empty summary.")
    return summary


def summarize_meeting_audio(
    audio_path: str,
    whisper_model_size: str = "small",
    llm_model: str = "qwen3.5:4b",
    llm_timeout_seconds: int = 600,
) -> dict:
    """End-to-end orchestration: audio -> transcript -> summary."""
    ensure_ollama_model(llm_model)
    stt_model = load_whisper_model(whisper_model_size)
    transcript = transcribe_audio(audio_path, stt_model)
    summary = summarize_transcript(
        transcript,
        llm_model=llm_model,
        request_timeout_seconds=llm_timeout_seconds,
    )

    return {
        "audio_path": str(Path(audio_path).expanduser().resolve()),
        "transcript": transcript,
        "summary": summary,
    }

In [ ]:
# Set this path to your meeting audio file, then run this cell.
AUDIO_FILE_PATH = "/absolute/path/to/meeting_audio.wav"

result = summarize_meeting_audio(
    audio_path=AUDIO_FILE_PATH,
    whisper_model_size="small",
    llm_model="qwen3.5:4b",
)

print("=== SUMMARY ===\n")
print(result["summary"])

# Optionally inspect the transcript:
# print(result["transcript"])

In [ ]:
# Optional: save output to JSON
output_path = Path("meeting_summary_output.json")
with output_path.open("w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"Saved output to: {output_path.resolve()}")